# Modelo estático: CAPM

En este ejercicio estimaremos el beta CAPM para varias acciones. El modelo CAPM 

In [ ]:
library(dplyr)
library(ggpubr)
library(tsibble)
library(feasts)
library(tidyquant)
library(ggplot2)
library(ggtime)
library(moments)

In [ ]:
pg<-tq_get("PG", get = "stock.prices",from="2022-01-01")|>select(date, adjusted)
pg<-pg|>mutate(day=ymd(date))|>as_tsibble(index=day)

nvda<-tq_get("NVDA", get = "stock.prices",from="2022-01-01")|>select(date, adjusted)
nvda<-nvda|>mutate(day=ymd(date))|>as_tsibble(index=day)

irx<-tq_get("^IRX", get = "stock.prices",from="2022-01-01")|>select(date, adjusted)
irx<-irx|>mutate(day=ymd(date))|>as_tsibble(index=day)

sp<-tq_get("^GSPC", get = "stock.prices",from="2022-01-01")|>select(date, adjusted)
sp<-sp|>mutate(day=ymd(date))|>as_tsibble(index=day)

In [ ]:
pg<-pg|>mutate(r=log(adjusted)-lag(log(adjusted)))
nvda<-nvda|>mutate(r=log(adjusted)-lag(log(adjusted)))
sp<-sp|>mutate(r=log(adjusted)-lag(log(adjusted)))

- Primero veamos el comportamiento de los precios

In [ ]:
pg.plot<-pg|>autoplot(adjusted)+labs(y="PG")
rpg.plot<-pg|>autoplot(r)+labs(y="r_PG")
nvda.plot<-nvda|>autoplot(adjusted)+labs(y="NVDA")
rnvda.plot<-pg|>autoplot(r)+labs(y="r_NVDA")
sp.plot<-sp|>autoplot(adjusted)+labs(y="S&P")
rsp.plot<-sp|>autoplot(r)+labs(y="r_S&P")
tbill.plot<-irx|>autoplot(adjusted)+labs(y="3 month TBill")

ggarrange(pg.plot,rpg.plot,
nvda.plot,rnvda.plot,
sp.plot, rsp.plot,
tbill.plot,
ncol=2,nrow=4)

- Agregamos a retornos mensuales

In [ ]:
pg_mes<-pg|>index_by(mes=~yearmonth(.))|>summarise(r=sum(r,na.rm=TRUE),.groups="drop")
nvda_mes<-nvda|>index_by(mes=~yearmonth(.))|>summarise(r=sum(r,na.rm=TRUE),.groups="drop")
sp_mes<-sp|>index_by(mes=~yearmonth(.))|>summarise(r=sum(r,na.rm=TRUE),.groups="drop")
irx_mes<-irx|>index_by(mes=~yearmonth(.))|>summarise(rf=
  0.01*(mean(adjusted,na.rm=TRUE)),.groups="drop")

- Creamos los excesos de retorno. Para ello debemos anualizar los retornos logaritmicos de PG y NVDA. Las tasas de los T-Bill están anualizadas. A partir de ello creamos el exceso de retorno

In [ ]:
pg_mes<-pg_mes|>mutate(r_a=r*12)
nvda_mes<-nvda_mes|>mutate(r_a=r*12)
sp_mes<-sp_mes|>mutate(r_a=r*12)

In [ ]:
rpg<-pg_mes$r_a-irx_mes$rf
rnvda<-nvda_mes$r_a-irx_mes$rf
rsp<-sp_mes$r_a-irx_mes$rf

In [ ]:
par(mfrow = c(3, 2), mar = c(4, 4, 3, 1))
plot(rpg,main="r_PG-r_f",type="l",lwd = 1.5)
acf(rpg, "r_PG-r_f")
plot(rnvda,main="r_NVDA-r_f",type="l",lwd = 1.5)
acf(rnvda, "r_NVDA-r_f")
plot(rsp,main="r_S&p-r_f",type="l",lwd = 1.5)
acf(rpg, "r_SP-r_f")
par(mfrow = c(1, 1))

- Corremos el modelo. Estimamos por MCO

In [ ]:
beta_pg<-lm(rpg~rsp)
summary(beta_pg)

In [ ]:
beta_nvda<-lm(rnvda~rsp)
summary(beta_nvda)

- Es común encontrar medidas obtenidas sin usar el exceso de retorno. Veamos la diferencia

In [ ]:
beta_pg_no_rf<-lm(pg_mes$r~sp_mes$r)
summary(beta_pg_no_rf)

In [ ]:
beta_nvda_no_rf<-lm(nvda_mes$r~sp_mes$r)
summary(beta_nvda_no_rf)

## Actividad

- Si fuera un asesor de inversiones, ¿que recomendaría basado en los valores $\hat{\beta}$ estimados?
- Para cada uno de los modelos, obtenga los residuales y realice la gráfica de ACF ¿Hay evidencia de autocorrelación serial? ¿Por qué es importante que los residuales no estén autocorrelacionados?
- Elija una acción diferente y replique el ejercicio
- Para cada uno de las acciones, calcule el ratio de Sharpe. Inteprete los valores y compare entre los activos